<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/02-rdd-fundamentals/02-core-transformations.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 2.2 — map, flatMap, filter, reduce, and the *ByKey family

Includes the `reduceByKey` vs `groupByKey` shuffle-size experiment —
have the Spark UI's **Stages** tab ready.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("2.2-core-transformations")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

lines = sc.textFile(f"{DATA_DIR}/sample_logs.txt")
lines.first()

## map vs flatMap — the classic confusion, settled in one cell

In [ ]:
as_map     = lines.map(lambda l: l.split())
as_flatmap = lines.flatMap(lambda l: l.split())

print("map:     count =", as_map.count(),     "| first =", as_map.first()[:4], "...")
print("flatMap: count =", as_flatmap.count(), "| first =", as_flatmap.first())

In [ ]:
# filter, and chaining
errors = lines.filter(lambda l: " ERROR " in l)
errors.collect()

## reduce — and why the function must be associative + commutative

In [ ]:
nums = sc.parallelize([4, 1, 7, 2])
print("sum:", nums.reduce(lambda a, b: a + b))

# Subtraction is NOT associative — result depends on partitioning:
print("1 partition: ", sc.parallelize([4, 1, 7, 2], 1).reduce(lambda a, b: a - b))
print("2 partitions:", sc.parallelize([4, 1, 7, 2], 2).reduce(lambda a, b: a - b))
print("4 partitions:", sc.parallelize([4, 1, 7, 2], 4).reduce(lambda a, b: a - b))

## Pair RDDs: count events per component

In [ ]:
pairs = lines.map(lambda l: (l.split()[3], 1))
pairs.take(5)

In [ ]:
print(sorted(pairs.reduceByKey(lambda a, b: a + b).collect()))
print(sorted(pairs.groupByKey().mapValues(len).collect()))   # same answer...

## ...but not the same shuffle. The experiment:

10M records, only 4 distinct keys — the best case for map-side combining.
Run both cells, then compare **Shuffle Read/Write** for the two jobs' first
stages in the UI (Stages tab). Expect KBs vs tens of MBs.

In [ ]:
big_pairs = sc.parallelize(range(10_000_000), 8).map(lambda i: (i % 4, 1))

big_pairs.reduceByKey(lambda a, b: a + b).collect()   # tiny shuffle: 4 keys x 8 partitions

In [ ]:
big_pairs.groupByKey().mapValues(len).collect()       # shuffles ALL 10M records

## aggregateByKey — mean latency per component

In [ ]:
def parse(l):
    parts = dict(kv.split("=") for kv in l.split()[4:] if "=" in kv)
    if "latency_ms" in parts:
        return (l.split()[3], int(parts["latency_ms"]))
    return None

lat = lines.map(parse).filter(lambda x: x is not None)

sum_cnt = lat.aggregateByKey(
    (0, 0),
    lambda acc, v: (acc[0] + v, acc[1] + 1),
    lambda a, b: (a[0] + b[0], a[1] + b[1]),
)
sorted(sum_cnt.mapValues(lambda t: round(t[0] / t[1], 1)).collect())

## Exercises

1. Word count, the classic: most frequent 5 tokens in the log file (`flatMap` → pair → `reduceByKey` → `takeOrdered(5, key=lambda kv: -kv[1])`).
2. Events per user (`user=uNNN` is the 4th space-separated field's neighbor — parse it), sorted descending by count.
3. Per-component MAX latency — which is the right tool: `reduceByKey` or `aggregateByKey`? Why?
4. Rewrite exercise 3's pipeline to also return the count of latency samples per component in the same pass (now which tool?).

In [ ]:
# your exercise code here
